# 02 - Flattening Layer (Phase 2)

Purpose: rebuild deterministic parquet contracts from read-only Mongo base collections.

Inputs: `players`, `bet_transactions`, `deposittransactions`, `withdrawaltransactions`, `bonustransactions`, `useractivitylogs`, `loginlogs`, plus identity/account support collections.

Outputs: parquet files and markdown reports under `data/`.

Core production logic lives in `src/frauddet/flatten.py`; this notebook only orchestrates, previews, and reconciles.

In [1]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from frauddet import config
from frauddet.flatten import OUTPUT_FILES, rebuild_all_flattened_outputs
from frauddet.identity import IdentityMapper, is_objectid_hex, normalize_phone
from frauddet.io_mongo import get_database

pd.set_option("display.max_columns", 80)

## Rebuild

In [2]:
result = rebuild_all_flattened_outputs(config.DATA_DIR)
result.raw_counts, result.parquet_counts, result.report_paths

({'players': 106,
  'bets': 433,
  'deposits': 1100,
  'withdrawals': 93,
  'bonus': 76,
  'activity': 4561,
  'logins': 1068},
 {'players': 106,
  'bets': 433,
  'money': 1149,
  'bonus': 76,
  'activity': 4561,
  'logins': 1068},
 {'unjoined': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/unjoined_report.md'),
  'reconciliation': WindowsPath('C:/Users/dev/OneDrive/Documents/Fraud Detection - FS Book/Fraud-Detection-FS-Book/data/flatten_reconciliation.md')})

## Schema Previews

In [3]:
frames = {
    name: pd.read_parquet(config.DATA_DIR / filename)
    for name, filename in OUTPUT_FILES.items()
}

for name, frame in frames.items():
    print(f"\n## {name}: {len(frame):,} rows")
    print(frame.dtypes.astype(str).to_string())
    display(frame.head(3))


## players: 106 rows
player_key                         str
phone                              str
created_at         datetime64[us, UTC]
kyc_status                         str
is_deleted                        bool
archived                          bool
referred_by_key                    str
nationality                        str
dob                                str
username_raw                       str


,player_key,phone,created_at,kyc_status,is_deleted,archived,referred_by_key,nationality,dob,username_raw
0,6a0ea9ff174ad3c431d9e16d,757575757,2026-05-21 06:45:19.652000+00:00,Approved,False,False,NaN,Uganda,2000-Jan-01,0757575757
1,6a0eabae174ad3c431d9e6b6,756363636,2026-05-21 06:52:30.870000+00:00,Approved,False,False,NaN,NaN,2000-Jan-01,0756363636
2,6a0eac61174ad3c431d9e947,751234567,2026-05-21 06:55:29.137000+00:00,Approved,False,False,NaN,NaN,1999-Dec-01,0751234567



## bets: 433 rows
player_key                          str
unjoined_class                      str
ticket_id                           str
game_type                           str
status                              str
result                              str
stake                             int64
stake_real                        int64
stake_bonus                       int64
is_free_bet                        bool
payout                          float64
potential_payout                float64
total_odds                      float64
currency                            str
source_currency                     str
created_at          datetime64[us, UTC]
settled_at          datetime64[us, UTC]
bet_type                            str
n_selections                      int64
min_part_odds                   float64
max_part_odds                   float64
sports                           object


,player_key,unjoined_class,ticket_id,game_type,status,result,stake,stake_real,stake_bonus,is_free_bet,payout,potential_payout,total_odds,currency,source_currency,created_at,settled_at,bet_type,n_selections,min_part_odds,max_part_odds,sports
0,6a215a2a7364bea0f4f5a90f,NaN,96910293-b16d-4641-b0dc-0e4e3300babd,Sports-book,SETTLED,WIN,1000,1000,0,False,1317.5,1550.0,1.55,UGX,INR,2026-05-26 05:32:50.833000+00:00,2026-05-26 20:26:20.812000+00:00,single,1,1.55,1.55,[football]
1,6a215a2a7364bea0f4f5a90f,NaN,c7e8283b-c509-43c1-8672-c1a216f93fb8,Sports-book,SETTLED,WIN,1000,1000,0,False,1258.0,1480.0,1.48,UGX,INR,2026-05-26 09:05:50.430000+00:00,2026-05-26 20:26:20.738000+00:00,single,1,1.48,1.48,[football]
2,NaN,unknown,6cd3fc1e-3954-41c7-b2fa-3731e02207e8,Sports-book,SETTLED,LOSE,1000,1000,0,False,0.0,2450.0,2.45,UGX,INR,2026-05-26 20:48:43.918000+00:00,2026-06-17 13:20:11.644000+00:00,single,1,2.45,2.45,[football]



## money: 1,149 rows
player_key                                  str
unjoined_class                              str
txn_type                                    str
transaction_id                              str
amount                                    int64
currency                                    str
payment_method                              str
account_number                              str
final_status                                str
requested_at                datetime64[us, UTC]
finalized_at                datetime64[us, UTC]
execution_type                              str
recipient_normalized                        str
is_third_party_recipient                   bool
bonus_tags                               object
is_money_in                                bool
is_money_out                               bool
is_pending_withdrawal                      bool


...

,player_key,unjoined_class,source_id,txn_type,amount,bonus_type,ref_trans_id,ref_kind,currency,created_at
0,6a215a2a7364bea0f4f5a90f,NaN,6a14589ab76d79e0209cf70e,ALLOCATED,800.0,First Deposit,TXNY_MPL7CMMP_7069B12A63A6,deposit,UGX,2026-05-25 14:11:38.818000+00:00
1,6a0eac61174ad3c431d9e947,NaN,6a1549beb76d79e020b515d8,ALLOCATED,800.0,First Deposit,TXNY_MPM9JG1A_8EC8872C1F32,deposit,UGX,2026-05-26 07:20:30.986000+00:00
2,6a215a2a7364bea0f4f5a90f,NaN,6a1601edb76d79e020bbe445,RELEASE,200.0,First Deposit,96910293-b16d-4641-b0dc-0e4e3300babd,bet,UGX,2026-05-26 20:26:21.379000+00:00



## activity: 4,561 rows
player_key                        str
unjoined_class                    str
source_id                         str
action                            str
client_ip                         str
page                              str
device_type                       str
created_at        datetime64[us, UTC]


,player_key,unjoined_class,source_id,action,client_ip,page,device_type,created_at
0,NaN,unknown,6a0ea89609b73d64e79ce37a,LOGOUT,80.227.165.206,/,desktop,2026-05-21 06:39:18.872000+00:00
1,6a0ea9ff174ad3c431d9e16d,NaN,6a0eaa09174ad3c431d9e185,LOGIN,80.227.165.206,/,desktop,2026-05-21 06:45:29.857000+00:00
2,6a0ea9ff174ad3c431d9e16d,NaN,6a0eaa3b174ad3c431d9e434,NAVIGATE,80.227.165.206,/dashboard/deposit,desktop,2026-05-21 06:46:19.872000+00:00



## logins: 1,068 rows
player_key                        str
unjoined_class                    str
source_id                         str
fingerprint                       str
user_type                         str
success                        object
failure_reason                 object
created_at        datetime64[us, UTC]


,player_key,unjoined_class,source_id,fingerprint,user_type,success,failure_reason,created_at
0,NaN,staff,6a0ea980174ad3c431d9de56,10c1df6db2524a9848b00cf27110cab3510e8d466d42a1...,MANAGER,True,None,2026-05-21 06:43:12.403000+00:00
1,6a0ea9ff174ad3c431d9e16d,NaN,6a0eaa09174ad3c431d9e183,48b4e69c379c8c7cdd2b63fb2776e21414e8a741d26045...,PLAYER,True,None,2026-05-21 06:45:29.105000+00:00
2,NaN,staff,6a0eaaa0174ad3c431d9e4c3,10c1df6db2524a9848b00cf27110cab3510e8d466d42a1...,MANAGER,True,None,2026-05-21 06:48:00.564000+00:00


## Row Counts

In [4]:
pd.DataFrame(
    {
        "raw_mongo_rows": pd.Series(result.raw_counts),
        "parquet_rows": pd.Series(result.parquet_counts),
    }
)

,raw_mongo_rows,parquet_rows
activity,4561.0,4561.0
bets,433.0,433.0
bonus,76.0,76.0
deposits,1100.0,NaN
logins,1068.0,1068.0
money,NaN,1149.0
players,106.0,106.0
withdrawals,93.0,NaN


## Player Join Rates

In [5]:
join_rows = []
for name, frame in frames.items():
    if "player_key" not in frame.columns:
        continue
    nulls = int(frame["player_key"].isna().sum())
    join_rows.append(
        {
            "table": name,
            "rows": len(frame),
            "null_player_key": nulls,
            "join_rate": None if len(frame) == 0 else 1 - nulls / len(frame),
        }
    )

pd.DataFrame(join_rows).sort_values("table")

,table,rows,null_player_key,join_rate
4,activity,4561,379,0.916904
1,bets,433,31,0.928406
3,bonus,76,7,0.907895
5,logins,1068,548,0.486891
2,money,1149,20,0.982594
0,players,106,0,1.000000


## Currency Magnitude Verification

In [6]:
bets_frame = frames["bets"]
money_frame = frames["money"]

raw_inr_bets = bets_frame[bets_frame["source_currency"].astype("string").str.upper() == "INR"].copy()
ugx_deposits = money_frame[
    (money_frame["txn_type"] == "DEPOSIT")
    & (money_frame["currency"].astype("string").str.upper() == "UGX")
].copy()
same_players = sorted(
    set(raw_inr_bets.loc[raw_inr_bets["player_key"].notna(), "player_key"])
    & set(ugx_deposits.loc[ugx_deposits["player_key"].notna(), "player_key"])
)

currency_rows = []
for player_key in same_players:
    player_bets = raw_inr_bets.loc[raw_inr_bets["player_key"] == player_key, "stake"].dropna().astype(float)
    player_deposits = ugx_deposits.loc[ugx_deposits["player_key"] == player_key, "amount"].dropna().astype(float)
    if player_bets.empty or player_deposits.empty:
        continue
    currency_rows.append(
        {
            "player_key": player_key,
            "n_bets": len(player_bets),
            "n_deposits": len(player_deposits),
            "median_source_inr_stake": player_bets.median(),
            "median_ugx_deposit": player_deposits.median(),
            "deposit_to_stake_median_ratio": player_deposits.median() / player_bets.median()
            if player_bets.median()
            else None,
        }
    )

currency_check = pd.DataFrame(currency_rows)
print(f"same_player_count={len(currency_check)}")
display(currency_check.sort_values(["n_bets", "n_deposits"], ascending=False).head(20))

stake_quantiles = raw_inr_bets["stake"].dropna().astype(float).quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 1])
deposit_quantiles = ugx_deposits.loc[ugx_deposits["player_key"].isin(same_players), "amount"].dropna().astype(float).quantile([0, 0.1, 0.25, 0.5, 0.75, 0.9, 1])
print("source INR bet stake quantiles")
print(stake_quantiles.to_string())
print("UGX deposit quantiles for same players")
print(deposit_quantiles.to_string())

bet_values = set(raw_inr_bets["stake"].dropna().astype(float).unique())
deposit_values = set(ugx_deposits.loc[ugx_deposits["player_key"].isin(same_players), "amount"].dropna().astype(float).unique())
overlap_values = sorted(bet_values & deposit_values)
print("overlapping stake/deposit denominations", overlap_values[:20])
print("flattened bet currencies", sorted(bets_frame["currency"].dropna().astype(str).unique()))
print("source bet currencies", sorted(bets_frame["source_currency"].dropna().astype(str).unique()))

if raw_inr_bets.empty or currency_check.empty:
    raise AssertionError("Currency verification requires INR-source bets and same-player UGX deposits.")

median_deposit_to_stake = deposit_quantiles.loc[0.5] / stake_quantiles.loc[0.5]
if 20 <= median_deposit_to_stake <= 45:
    raise AssertionError(
        "INR bet amounts look FX-scaled versus UGX deposits; stop before relabeling. "
        f"median deposit/stake ratio={median_deposit_to_stake:.2f}"
    )
if not overlap_values:
    raise AssertionError("No shared denominations between source-INR stakes and UGX deposits; stop before relabeling.")

print(
    "Outcome: source INR bet amounts are UGX-like denominations, not ~31x FX-scaled; "
    "currency normalized to UGX by relabel only."
)


same_player_count=19


,player_key,n_bets,n_deposits,median_source_inr_stake,median_ugx_deposit,deposit_to_stake_median_ratio
5,6a21639b7364bea0f4f5b356,237,37,1000.0,1000.0,1.000000
0,6a0ea9ff174ad3c431d9e16d,31,6,1100.0,13050.0,11.863636
6,6a2170b97364bea0f4f5c024,21,5,1321.0,3600.0,2.725208
8,6a26821a305ac95f848e7e35,14,5,1000.0,6100.0,6.100000
1,6a0eac61174ad3c431d9e947,13,5,1000.0,2600.0,2.600000
11,6a26b32a3eaf5b91c9200d3c,10,2,1000.0,25500.0,25.500000
2,6a0eac6b174ad3c431d9e988,9,7,1000.0,10000.0,10.000000
3,6a215a2a7364bea0f4f5a90f,9,7,1000.0,10000.0,10.000000
4,6a2161017364bea0f4f5aede,9,3,1100.0,2100.0,1.909091
9,6a26959dfc099dd782863ecf,8,2,1000.0,10000.0,10.000000


source INR bet stake quantiles
0.00      1000.0
0.10      1000.0
0.25      1000.0
0.50      1000.0
0.75      1000.0
0.90      1224.8
1.00    100000.0
UGX deposit quantiles for same players
0.00         50.0
0.10        100.0
0.25       1000.0
0.50       2944.0
0.75      10000.0
0.90     100000.0
1.00    3000000.0
overlapping stake/deposit denominations [np.float64(1000.0), np.float64(1100.0), np.float64(2000.0), np.float64(2100.0), np.float64(3100.0), np.float64(100000.0)]
flattened bet currencies ['UGX']
source bet currencies ['INR']
Outcome: source INR bet amounts are UGX-like denominations, not ~31x FX-scaled; currency normalized to UGX by relabel only.


## Hardening Checks

In [7]:
print("result.anomalies", result.anomalies)
print("result.identity_collisions", result.identity_collisions)
print("result.collapsed_withdrawal_count", result.collapsed_withdrawal_count)
frames["money"].loc[frames["money"]["txn_type"].eq("DEPOSIT"), "final_status"].value_counts(dropna=False)


result.anomalies []
result.identity_collisions [{'phone': '751452653', 'winning_key': '6a32813d3bf6ed597b7418fd', 'losing_keys': ('6a3261c5aa7ed994dfdadcb3', '6a32762c1780aec518efd60c')}]
result.collapsed_withdrawal_count 49


final_status
initiated                853
completed                237
manual_reconciliation      6
failed                     4
Name: count, dtype: int64

## referredBy Verification

In [8]:
with get_database() as db:
    player_docs = list(
        db[config.get_collection_names()["players"]].find(
            {},
            {
                "_id": 1,
                "username": 1,
                "contactNo": 1,
                "playerId": 1,
                "referralCode": 1,
                "referredBy": 1,
                "referredCode": 1,
                "isDeleted": 1,
                "createdAt": 1,
            },
        )
    )

player_ids = {str(doc.get("playerId")) for doc in player_docs if doc.get("playerId") is not None}
referral_codes = {str(doc.get("referralCode")) for doc in player_docs if doc.get("referralCode")}
player_keys = {str(doc.get("_id")) for doc in player_docs if doc.get("_id") is not None}
phones = {
    phone
    for doc in player_docs
    for phone in (normalize_phone(doc.get("username")), normalize_phone(doc.get("contactNo")))
    if phone
}
mapper = IdentityMapper.from_players(player_docs)

def referred_by_format(value):
    text = str(value).strip()
    if text in player_ids:
        return "playerId"
    if text in referral_codes:
        return "referralCode"
    if text in player_keys or is_objectid_hex(text):
        return "players._id_hex"
    if normalize_phone(text) in phones:
        return "phone"
    return "unknown"

referred_by_rows = []
for doc in player_docs:
    value = doc.get("referredBy")
    if value is None:
        continue
    resolved = mapper.resolve(value, allow_player_id=True, allow_referral_code=True)
    referred_by_rows.append(
        {
            "player_key": str(doc.get("_id")),
            "username": doc.get("username"),
            "referredBy": value,
            "referredCode": doc.get("referredCode"),
            "format": referred_by_format(value),
            "resolved_key": resolved.player_key,
        }
    )

referred_by_df = pd.DataFrame(
    referred_by_rows,
    columns=["player_key", "username", "referredBy", "referredCode", "format", "resolved_key"],
)
print(f"non_null_referredBy={len(referred_by_df)}")
if referred_by_df.empty:
    print("No non-null players.referredBy values in current dev Mongo; decision evidence remains proposed.")
else:
    display(referred_by_df)
    display(
        referred_by_df.groupby(["format", "resolved_key"], dropna=False)
        .size()
        .reset_index(name="count")
    )


non_null_referredBy=2


,player_key,username,referredBy,referredCode,format,resolved_key
0,6a0f0df7a5d7ce78c657a62f,0751212121,10003905,Y6YCDR,playerId,6a0ef9cca5d7ce78c65765d9
1,6a14351b6ac0a3933202239e,0757676767,10003905,Y6YCDR,playerId,6a0ef9cca5d7ce78c65765d9


,format,resolved_key,count
0,playerId,6a0ef9cca5d7ce78c65765d9,2


## Reconciliation

In [9]:
print((config.DATA_DIR / "flatten_reconciliation.md").read_text(encoding="utf-8"))

# Flatten Reconciliation

| Source | Raw Mongo rows | Parquet rows | Delta explanation |
| --- | ---: | ---: | --- |
| players | 106 | 106 | One row per player; no drops. |
| bets | 433 | 433 | One row per ticket; unjoined rows kept. |
| deposits | 1100 | included in money | No deposit deduplication; all statuses kept and flags determine money-in. |
| withdrawals | 93 | 49 collapsed groups | Lifecycle collapse by `transactionId`; status order completed > declined > failed > pending > initiated. |
| money | 1100 deposits + 49 collapsed withdrawals | 1149 | ASSERTED: money rows equal deposits plus collapsed withdrawals. |
| bonus | 76 | 76 | One row per bonus transaction; currency is assumed `UGX` because source has no currency field. |
| activity | 4561 | 4561 | One row per activity event; unjoined rows kept. |
| logins | 1068 | 1068 | One row per login log; staff rows kept with null player_key. |

## Withdrawal Account Anomaly

Status x `toAccountId` resolution:

| status | cashaccount

Expected deltas explained above:

- Withdrawals collapse from lifecycle documents to one row per `transactionId`.
- Deposits are not deduplicated; all statuses are kept and interpreted through boolean flags.
- Unjoined source rows are kept with `player_key = NULL` and `unjoined_class`.
- Bonus currency is assumed from `config.yaml` because `bonustransactions` has no currency field.

There are zero unexplained drops in the current run.

## Reports

In [10]:
for name, path in result.report_paths.items():
    print(f"\n# {name}: {path}\n")
    print(path.read_text(encoding="utf-8"))


# unjoined: C:\Users\dev\OneDrive\Documents\Fraud Detection - FS Book\Fraud-Detection-FS-Book\data\unjoined_report.md

# Unjoined Report

Rows are kept in parquet with `player_key = NULL` and excluded later at feature time.

Known orphan-wallet IDs called out from Phase 1b:
- `0751231231`
- `0754321012`

## Identity Collisions

| phone | winning_key | losing_keys |
| --- | --- | --- |
| 751452653 | 6a32813d3bf6ed597b7418fd | 6a3261c5aa7ed994dfdadcb3, 6a32762c1780aec518efd60c |

## bets

| unjoined_class | count | sample IDs |
| --- | ---: | --- |
| pre_registration | 16 | 17759968-d60a-4a98-b73a-f99923f8f4aa, 797ab76e-19bb-4648-9d46-46eac29c7a5e, ca52d46a-0d45-4bdf-b894-b3cfc5184b34, 7a7c39c4-82eb-4d06-ba96-bfbb66b9d0f3, 40fd0c8c-e00c-490c-a0fd-aa465ab9784f, 870cf27a-dbf8-4660-adc9-853881d402bb, 5efc8ac3-2d21-44b0-af8a-a814680c4090, d9a2bf87-c468-40b7-b1fb-88338440220d, 00e6b9d7-98c6-49be-b11a-efa18ff2e001, 2e481ba3-a524-4bb1-a371-12d774658f6a |
| unknown | 15 | 6cd3fc1e-3954-41c7-b2f

## FINDINGS — Notebook 02: flattening layer — 2026-06-18
- Verified: Executed Phase 2 run produced `players=106`, `bets=433`, `money=1,149`, `bonus=76`, `activity=4,561`, `logins=1,068` parquet rows from current Mongo. Money rows reconcile as 1,100 deposits plus 49 collapsed withdrawal transaction groups from 93 lifecycle docs.
- Verified: Reconciliation report visibly includes `Coded Invariants` with both withdrawal invariants VERIFIED and an `Identity Collisions` section. The hardening cell prints `result.anomalies=[]`, `result.identity_collisions=[{'phone': '751452653', ...}]`, `result.collapsed_withdrawal_count=49`, and deposit status counts including `manual_reconciliation=6`.
- Verified: Login unjoined rows are classified as `staff`: current unjoined login classes are `staff=502`, `pre_registration=22`, `unknown=24`.
- Verified: `players.referredBy` has two non-null current values; both are `playerId` references (`10003905`) and resolve to `6a0ef9cca5d7ce78c65765d9` via explicit player-profile opt-in.
- Verified: Currency check found 19 same-player source-INR-bet/UGX-deposit comparators. Source-INR stake median is `1000`, same-player UGX deposit median is `2944`, denominations overlap (`1000`, `1100`, `2000`, `2100`, `3100`, `100000`), and there is no ~31x median signal. Bets are flattened to `currency=UGX` with `source_currency=INR`; no FX math is applied.
- Verified: Product-owner decisions are applied in Phase 2 contracts: 6 `manual_reconciliation` deposits are all `is_money_in=True`; bets expose `settled_at` and no longer expose `settled_at_proxy`.
- Surprises: Current live counts changed again while Phase 2 was being closed, and there is now one normalized-phone identity collision (`751452653`) resolved by the mapper's deterministic non-deleted/latest-created rule.
- Gaps: Bonus transactions still have no source currency field; Phase 2 assigns the production default `UGX`. `loginlogs.operationType == "Login"` remains an assumed success signal until confirmed.
- Decisions needed: Confirm who `0751231231` / `0754321012` are, confirm whether pre-registration identities can transact in production, and confirm login success semantics.
- Updated assumptions: Later phases should read only flattened parquet contracts, can use bets↔money ratios after UGX-normalized flattened outputs, should treat casino betting features as null/sparse by v1 scope, and should not start casino game-exploitation features without new per-round logging and a product scope change.
